# PRICING DINÁMICO DE HABITACIONES DE HOTEL

*Cristian Rubio Barato*

*Francisco Martínez Esteso*

*José Vicente García López*

*Víctor Ortega Gómez*


## ÍNDICE

1. [PREPROCESAMIENTO](#1-preprocesamiento)    
   1.1 [Instalar MLRun](#11-instalar-mlrun)   
   1.2 [Crear proyecto](#12-crear-proyecto)   
   1.3 [Preprocesamiento dataset](#13-preprocesamiento-dataset)   

2. [ENTRENAMIENTO](#2-entrenamiento)  
   2.1 [Funcion entrenamiento](#21-funcion-entrenamiento)   
   2.2 [Resumen entrenamiento](#22-resumen-entrenamiento)   

3. [DESPLIEGUE E INFERENCIA](#3-despliegue-e-inferencia)  

In [1]:
random_state = 27912

## 1. PREPROCESAMIENTO

### 1.1 Instalar MLRun

In [2]:
# Install MLRun and sklearn, run this only once (restart the notebook after the install !!!)
%pip install mlrun scikit-learn~=1.5.1 numpy~=1.26.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 10.4 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.23.3
    Uninstalling numpy-1.23.3:
      Successfully uninstalled numpy-1.23.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.56.2 requires numpy<1.24,>=1.18, but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### 1.2 Crear proyecto

In [1]:
import mlrun

In [2]:
project = mlrun.get_or_create_project("pricing-hotel", "./", user_project=True)

> 2025-04-18 15:50:44,034 [info] Created and saved project: {"context":"./","from_template":null,"name":"pricing-hotel-jovyan","overwrite":false,"save":true}
> 2025-04-18 15:50:44,035 [info] Project created successfully: {"project_name":"pricing-hotel-jovyan","stored_in_db":true}


### 1.3 Preprocesamiento dataset

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import StandardScaler
import mlrun

In [4]:
df = pd.read_csv('hotel_booking.csv')

df = df.dropna(subset=["adr"])
df.fillna(0, inplace=True)

categorical_vars = df.select_dtypes(include=['object', 'category']).columns.tolist()
continuous_vars = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Conversión los valores de children, arrival_date_month a enteros. reservation_status_date lo cambiaremos a datetime. 
# Estos cambios se harán para poder ser manejados correctamente en el dataset. 

df["reservation_status_year"] = pd.to_datetime(df["reservation_status_date"]).dt.year
df["arrival_date_month"] = pd.to_datetime(df["arrival_date_month"], format = "%B").dt.month
df["reservation_status_date"] = pd.to_datetime(df["reservation_status_date"]).dt.date

df["children"] = df["children"].astype(int)

# Eliminamos los valores del dataframe que tienen ruido respecto al ADR (esto se explica en los hitos anteriores).

start_date = datetime.strptime("2015-07", "%Y-%m").date()

df['reservation_status_date'] = pd.to_datetime(df['reservation_status_date']).dt.date  

df_filtered = df[df['reservation_status_date'] >= start_date]

# Mantendremos solo las columnas relevantes y eliminamos 
# las columnas con datos personales y con datos que no influyan para predecir el ADR

columns_to_drop = [
    "days_in_waiting_list", "name", "email", 
    "phone-number", "credit_card", "agent", "company", 
    "booking_changes", "arrival_date_week_number"
]

df.drop(columns=columns_to_drop, inplace=True, errors="ignore")

# También eliminaremos las filas que no tienen huespedes si hubiese alguna ya que no nos interesan.

df = df[~((df["adults"] == 0) & (df["children"] == 0) & (df["babies"] == 0))]

# Tratamiento de los valores extremos de ADR que pueden afectar a las predicciones. 

Q1 = np.percentile(df["adr"], 25)
Q3 = np.percentile(df["adr"], 75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df = df[(df["adr"] >= lower_bound) & (df["adr"] <= upper_bound)]

# Transformaremos las variables categóricas en valores que el modelo de ML pueda entender. 
# Para ello aplicaremos la técnica de One-Hot Encoding.

categorical_vars = [col for col in categorical_vars if col not in columns_to_drop]

df = pd.get_dummies(df, columns=categorical_vars, drop_first=True)

# Realizaremos un escalado de las variables para que estén todas en la misma escala

from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

In [9]:
@mlrun.handler(outputs=["dataset", "label_column"])
def hotel_booking_preprocessor(context):
    # 1. Carga del CSV
    df = pd.read_csv('Pricing_Hotel/hotel_booking.csv')
    
    # 2. Filtrado y relleno de nulos
    df = df.dropna(subset=["adr"])
    df.fillna(0, inplace=True)
    
    # 3. Conversión de fechas y tipos
    df["reservation_status_date"] = pd.to_datetime(df["reservation_status_date"])
    df["reservation_status_year"]  = df["reservation_status_date"].dt.year
    df["reservation_status_date"]  = df["reservation_status_date"].dt.date
    df["arrival_date_month"]       = pd.to_datetime(df["arrival_date_month"], format="%B").dt.month
    df["children"] = df["children"].astype(int)
    
    # 4. Filtrado por fecha de inicio
    start_date = datetime.strptime("2015-07", "%Y-%m").date()
    df = df[df['reservation_status_date'] >= start_date]
    
    # 5. Eliminación de columnas irrelevantes
    columns_to_drop = [
        "days_in_waiting_list", "name", "email", "phone-number",
        "credit_card", "agent", "company", "booking_changes",
        "arrival_date_week_number"
    ]
    df.drop(columns=columns_to_drop, inplace=True, errors="ignore")
    
    # 6. Eliminación de filas sin huéspedes
    df = df[~((df["adults"] == 0) & (df["children"] == 0) & (df["babies"] == 0))]
    
    # 7. Tratamiento de outliers en ADR
    Q1 = np.percentile(df["adr"], 25)
    Q3 = np.percentile(df["adr"], 75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df = df[(df["adr"] >= lower_bound) & (df["adr"] <= upper_bound)]
    
    # 8. One‑Hot Encoding de categóricas
    categorical_vars = df.select_dtypes(include=['object', 'category']).columns.tolist()
    df = pd.get_dummies(df, columns=categorical_vars, drop_first=True)
    
    # 9. Escalado de numéricas
    numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns
    scaler = StandardScaler()
    df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
    
    # (Opcional) Guardar el scaler como artifact para usar en inferencia
    context.log_artifact("scaler", body=scaler, local_path="scaler.pkl")
    
    return df, "adr"

In [ ]:
# Registrar la función en MLRun
hotel_prep_fn = project.set_function(
    src="hotel_prep.py",                  
    name="hotel-prep",                    
    kind="job",                           
    image="mlrun/mlrun",                 
    handler="hotel_booking_preprocessor"  
)

project.save()


In [ ]:
hotel_pricing_run = project.run_function(name="hotel-prep", local=True)


In [ ]:
gen_data_run.state()

In [ ]:
gen_data_run.outputs

## 2. ENTRENAMIENTO

### 2.1 Funcion entrenamiento

In [ ]:
trainer = mlrun.import_function("hub://auto_trainer")

In [ ]:
metrics = {
    "R2": "r2",
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "RMSE": "neg_root_mean_squared_error",
    "Max Error": "max_error"
}

# Ejecutamos el paso de entrenamiento con:
trainer_run = project.run_function(
    trainer,
    handler="train",
    inputs={"dataset": hotel_pricing_run.outputs["dataset"]},
    params={
        "model_class": "sklearn.ensemble.RandomForestRegressor",
        "train_test_split_size": 0.2,
        "label_columns": "adr",
        "model_name": "hotel_price_rf",
        # Le decimos al auto_trainer qué métricas calcular
        "_metrics": metrics,  
    },
    # Buscar hiperparámetros y elegir el mejor por R2 (aunque ya lo hicimos en el hito 3):
    hyperparams={
        "n_estimators": [100, 200, 500],
        "max_depth": [10, 20, None],
        "min_samples_split": [2, 5, 10],
    },
    selector="max.R2",
    local=True
)

### 2.2 Resumen entrenamiento

In [ ]:
trainer_run.outputs

In [ ]:
trainer_run.metrics

## 3. DESPLIEGUE E INFERENCIA

In [ ]:
serving_fn = project.set_function(
    func="",
    name="serving",
    image="mlrun/mlrun",
    kind="serving",
    requirements=["scikit-learn~=1.5.1"]
)

In [ ]:
serving_fn.add_model(
    name="hotel-pricing-model",  
    model_path=trainer_run.outputs["model"],  
    class_name="mlrun.frameworks.sklearn.SklearnModelServer" 
)

In [ ]:
serving_fn.spec.graph.plot(rankdir="LR")

In [ ]:
server = serving_fn.to_mock_server()

In [ ]:
# Verificamos que el endpoint está listo
server.test("/v2/models/", method="GET")

In [ ]:
import numpy as np
import pandas as pd

# Como nuestro dataset después del preprocesamiento tiene 1167 columnas, 
# la inferencia la hacemos de forma aleatoria

# Usamos el mismo orden de columnas que en el dataset final
column_names = df.columns.tolist()
example_row = []

for col in df.dtypes.index:
    dtype = df.dtypes[col]

    if dtype == "bool":
        value = bool(np.random.randint(0, 2))  
    elif dtype == "int32" or dtype == "int64":
        value = int(np.random.randint(-2, 2))  
    elif dtype == "float64":
        value = float(np.random.normal(0, 1)) 
    else:
        value = 0

    example_row.append(value)

# Construimos el input que espera el modelo
example_input = {
    "inputs": [example_row]
}


In [ ]:
response = server.test("/v2/models/hotel-pricing-model/infer", body=example_input)
print(response)

In [ ]:
project.deploy_function(serving_fn)

response = serving_fn.invoke("/v2/models/hotel-pricing-model/infer", body=example_input)

print("Predicción del precio (adr):", response)